In [0]:
%pip install langchain langchain-core databricks-langchain faiss-cpu sentence-transformers pandas requests mlflow
dbutils.library.restartPython()


In [0]:
# ============================================================================
# IMPORTS & MLFLOW CONFIGURATION
# ============================================================================
import json
import requests
from urllib.parse import quote

import faiss
import mlflow
from sentence_transformers import SentenceTransformer

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from databricks_langchain import ChatDatabricks

mlflow.langchain.autolog()
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/covid_rag_experiments")

print("✅ Libraries imported and MLflow configured")


In [0]:
# ============================================================================
# LOAD VECTOR STORE & DOCUMENT CHUNKS (the internal COVID-19 knowledge base)
# ============================================================================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.read_index("/Volumes/main/covid_rag/covid_docs/faiss_index.bin")
chunk_df = spark.table("main.covid_rag.covid_chunk_embeddings").toPandas()

print(f"✅ FAISS index loaded — {len(chunk_df)} document chunks indexed")


In [0]:
# ============================================================================
# INITIALIZE THE LLM
# ============================================================================
# Note: tool calling requires a Databricks endpoint that supports function
# calling (e.g. Llama 3.1/3.3 instruct models on Foundation Model APIs).
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=600,
)

print("✅ LLM initialized: databricks-meta-llama-3-3-70b-instruct")


In [0]:
# ============================================================================
# TOOL 1 — INTERNAL: semantic search over the COVID-19 document knowledge base
# ============================================================================
def _internal_search(query: str, top_k: int = 4):
    """Run the actual FAISS lookup and return both LLM-readable text and
    structured citations (used for the sources shown to the user)."""
    q_emb = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    results = chunk_df.iloc[indices[0]][["source_file", "page_num", "chunk_text"]].copy()

    if results.empty:
        return {"text": "No relevant internal COVID-19 documents were found for this query.",
                "sources": []}

    context = "\n\n".join(
        f"Source: {row.source_file} (page {row.page_num})\n{row.chunk_text}"
        for row in results.itertuples()
    )
    sources = results[["source_file", "page_num"]].drop_duplicates().to_dict("records")
    return {"text": context, "sources": sources}


@tool
def search_covid_documents(query: str) -> str:
    """Search the internal COVID-19 knowledge base (medical PDFs covering
    symptoms, transmission, prevention, vaccines, treatment, variants,
    statistics, etc.). Use this for ANY question about COVID-19 / coronavirus."""
    return _internal_search(query)["text"]


In [0]:
# ============================================================================
# TOOL 2 — EXTERNAL: general knowledge lookup (Wikipedia) for non-COVID topics
# ============================================================================
def _external_search(query: str):
    """Run the actual Wikipedia lookup and return both LLM-readable text and
    structured citations."""
    headers = {"User-Agent": "covid-rag-project/1.0"}
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(query.strip())}"

    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            data = r.json()
            extract = data.get("extract", "").strip()
            page_url = data.get("content_urls", {}).get("desktop", {}).get("page", "")
            title = data.get("title", query)
            if extract:
                return {
                    "text": f"Wikipedia ({title}): {extract}",
                    "sources": [{"title": title, "url": page_url, "type": "wikipedia"}]
                }
    except Exception:
        pass

    return {"text": "No matching external reference article was found. "
                     "Answer using your own general knowledge if you can, "
                     "and say so if you're not confident.",
            "sources": []}


@tool
def search_external(query: str) -> str:
    """Search external general knowledge (Wikipedia) for any question that is
    NOT about COVID-19 — general facts, other topics, current entities, etc."""
    return _external_search(query)["text"]


In [0]:
# ============================================================================
# AGENT SETUP: system prompt + bound tools
# ============================================================================
SYSTEM_PROMPT = """You are a COVID-19 assistant. Your main purpose is to help
people with COVID-19-related questions (symptoms, transmission, prevention,
vaccines, treatment, variants, statistics, etc.) using the internal document
tool below. You can also answer other questions using the external tool, but
COVID-19 is your core focus.

When you answer directly without using a tool (e.g. greetings or small talk),
introduce yourself as a COVID-19 assistant — do not describe yourself as a
general-purpose helper. For example, greet the user and mention you're here
to help with COVID-19 questions, not "anything I can help with."

You have two tools available:
- search_covid_documents: use this for ANY question about COVID-19 / coronavirus
  (symptoms, transmission, prevention, vaccines, treatment, variants, statistics, etc).
- search_external: use this for questions that are NOT about COVID-19 — general
  knowledge, other topics, current facts, etc.

Guidelines:
- Decide for yourself, per question, which tool (if any) is needed. You may call
  a tool, call both if genuinely relevant, or call none at all (e.g. for greetings,
  small talk, or questions you can answer directly from the conversation history).
- Base your answer only on the tool results you receive plus the conversation
  history — do not invent facts, and do not fabricate document contents.
- If a tool returns nothing useful, say so briefly and plainly instead of guessing.
- Answer naturally and conversationally. Do not use numbered lists, headings, or
  meta-commentary about which tool you used — the application adds that labeling
  separately.
- When you answer a non-COVID-19 question using search_external, keep the
  answer itself brief and on-topic, then gently steer the conversation back
  to COVID-19 afterward — e.g. ask if they have any COVID-19 related
  questions you can help with. Don't do this after greetings or after
  COVID-19 answers, only after external-topic answers.
"""

tools = [search_covid_documents, search_external]
tool_registry = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

print("✅ Agent ready with tools:", list(tool_registry.keys()))


In [0]:
# ============================================================================
# AGENT LOOP: lets the LLM decide which tool(s) to call, executes them, and
# feeds results back until it produces a final answer.
# ============================================================================
def run_agent(question: str, history: list, max_steps: int = 3):
    """
    Run one turn of the agent.

    Args:
        question (str): the user's latest message
        history (list): prior conversation as LangChain BaseMessage objects
        max_steps (int): max number of tool-calling rounds before forcing a
                          final answer (safety limit against loops)

    Returns:
        dict: {
            "answer": str,              # raw answer text (no label yet)
            "tools_used": list[str],    # distinct tool names actually called
            "sources": list,            # structured citations from tool calls
            "messages": list,           # updated running message list
        }
    """
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + history + [HumanMessage(content=question)]

    tools_used = []
    sources = []

    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not getattr(response, "tool_calls", None):
            return {"answer": response.content, "tools_used": tools_used,
                    "sources": sources, "messages": messages}

        for call in response.tool_calls:
            name, args, call_id = call["name"], call["args"], call["id"]
            query = args.get("query", question)

            if name == "search_covid_documents":
                result = _internal_search(query)
            elif name == "search_external":
                result = _external_search(query)
            else:
                result = {"text": f"Unknown tool: {name}", "sources": []}

            tools_used.append(name)
            sources.extend(result["sources"])
            messages.append(ToolMessage(content=result["text"], tool_call_id=call_id))

    # Safety net: if the model is still trying to call tools after max_steps,
    # force a final answer without tool access.
    final = llm.invoke(messages)
    return {"answer": final.content, "tools_used": tools_used,
            "sources": sources, "messages": messages}


In [0]:
# ============================================================================
# Attach a source label based on which tool(s) the agent actually used —
# this is deterministic (not left to the LLM), so it's always accurate.
# ============================================================================
def label_for(tools_used: list) -> str:
    used = set(tools_used)
    if used == {"search_covid_documents"}:
        return "📄 **[Internal — COVID-19 Documents]**\n\n"
    if used == {"search_external"}:
        return "🌐 **[External Source]**\n\n"
    if used == {"search_covid_documents", "search_external"}:
        return "📄🌐 **[Internal + External Sources]**\n\n"
    return ""  # no tools called (e.g. greeting / direct chat-history answer)


def answer_query(question: str, chat_history: list = None) -> dict:
    """
    Full agentic turn: run the agent, then label the answer with its source.

    Returns:
        dict: {"answer": labeled answer text, "tools_used": [...], "sources": [...]}
    """
    if chat_history is None:
        chat_history = []

    try:
        result = run_agent(question, chat_history)
    except Exception as e:
        return {"answer": f"Sorry, something went wrong answering that: {e}",
                "tools_used": [], "sources": []}

    label = label_for(result["tools_used"])
    return {
        "answer": label + result["answer"],
        "tools_used": result["tools_used"],
        "sources": result["sources"],
    }


# 🤖 COVID-19 Agentic Assistant (Internal + External Tool-Calling)

## ✅ Using the Chatbot

Run the cell below to initialize the chat interface, then use these commands
in **any new cell**:

```python
chat("What is COVID-19?")               # agent calls search_covid_documents
chat("What are the symptoms?")          # follow-up, uses conversation history
chat("Who is the Prime Minister of Canada?")  # agent calls search_external
chat("hello!")                          # agent answers directly, no tool call

show_history()
clear_history()
```

## 🎯 How it works

There's no hardcoded greeting list or keyword/distance routing rules. The LLM
itself is given two tools — `search_covid_documents` (internal FAISS index)
and `search_external` (Wikipedia) — and decides, per question, whether to
call one, both, or neither. Every answer is labeled with the source(s) it
actually used:

* 📄 **Internal** — answered from the COVID-19 document knowledge base
* 🌐 **External** — answered from Wikipedia / general knowledge
* 📄🌐 **Internal + External** — the agent used both
* *(no label)* — answered directly, e.g. greetings or "what did I just ask?"

---


In [0]:
# ============================================================================
# SIMPLE CHAT INTERFACE FOR DATABRICKS NOTEBOOK
# ============================================================================
conversation = []  # list of LangChain BaseMessage objects (running memory)

def chat(question: str):
    """Ask the agent a question; keeps conversation memory across calls."""
    global conversation

    result = answer_query(question, chat_history=conversation)

    conversation.append(HumanMessage(content=question))
    conversation.append(AIMessage(content=result["answer"]))

    print(f"\n{result['answer']}")
    if result["sources"]:
        print(f"\nSources: {json.dumps(result['sources'], indent=2)}")

    return result


def clear_history():
    """Reset conversation memory."""
    global conversation
    conversation = []
    print("✅ Chat history cleared!")


def show_history():
    """Print the current conversation."""
    if not conversation:
        print("No chat history yet.")
        return
    for msg in conversation:
        role = "USER" if isinstance(msg, HumanMessage) else "ASSISTANT"
        content = msg.content[:120] + "..." if len(msg.content) > 120 else msg.content
        print(f"  {role}: {content}")


print("✅ COVID-19 agentic chatbot ready. Try: chat('What is COVID-19?')")


In [0]:
chat("What is COVID-19 and how does it spread?")


In [0]:
chat("What are the symptoms of it?")


In [0]:
chat("What is NLP?")


In [0]:
chat("hey, how are you?")


In [0]:
chat("what was my last question?")